# Plant disease classification with transfer learning

In this project, I investigate how ImageNet pretraining affects plant disease classification when labeled training data is limited. The comparison uses randomly initialized and ImageNet-pretrained ResNet-18 models with the same nominal 10% training subset and validation set.

The implementation has been tested on synthetic CPU data. No PlantVillage GPU experiment results are recorded in this notebook yet. The published test split is reserved for final evaluation.

## 1. Set up Colab

This notebook opens through **Colab → File → Open notebook → GitHub** using `Ifethecoder/plant-disease-classifier`.

1. Under **Runtime → Change runtime type**, select a **GPU** and runtime version **2026.07**, which uses Python 3.12. Other runtime versions need to use Python 3.12 as well. Available versions are listed in the [Colab runtime documentation](https://research.google.com/colaboratory/runtime-version-faq.html).
2. Run the setup cell below once to install dependencies.
3. Use **Runtime → Restart session** so the session loads the newly installed libraries.
4. Rerun the setup cell to restore its variables and project path, then run the remaining cells in order. An unchanged installation does not need a second restart.

Alternatively, an uploaded source ZIP can supply the project files, with `PROJECT_DIR` pointing to the extracted root. The setup cell stops before installation if the Python version does not match. The next code cell stops before any dataset download if a GPU is unavailable.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(
        "This Colab demo uses Python 3.12. Under Runtime → Change runtime type, "
        "select runtime version 2026.07 (Python 3.12), then reconnect."
    )

PROJECT_DIR = Path.cwd() if (Path.cwd() / "src/dataset.py").exists() else Path.cwd() / "plant-disease-classifier"
if not (PROJECT_DIR / "src/dataset.py").exists():
    subprocess.run(["git", "clone", "https://github.com/Ifethecoder/plant-disease-classifier.git",
                    str(PROJECT_DIR)], check=True)
PROJECT_DIR = PROJECT_DIR.resolve()
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
print("First installation: restart the session, rerun this cell, then continue.")

## 2. Choose settings and storage

Both models use Adam with learning rate 0.001, batch size 32 and five epochs. All layers are fine-tuned, with identical ImageNet normalization. Outputs are saved to Drive to survive a Colab disconnection, and each run gets a new directory. With Drive disabled, results remain in temporary runtime storage and need to be downloaded before disconnecting.

In [ ]:
import json
import tempfile
from datetime import datetime, timezone

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
CACHE_DIR = Path(tempfile.gettempdir()) / "plant-village-cache"
os.environ["TORCH_HOME"] = str(CACHE_DIR / "torch")

import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "A GPU is required for this Colab demo. Select Runtime → Change runtime type → GPU, "
        "then rerun setup after reconnecting."
    )
DEVICE = "cuda"
print("GPU:", torch.cuda.get_device_name(0))

import torchvision
from src.dataset import load_metadata, make_splits, load_images, make_loaders, seed_everything
from src.models import build_model, load_model, predict_image
from src.train import train_model
from src.evaluate import evaluate, plot_history, plot_predictions

FRACTION = 0.10  # Later experiments: 0.01, 0.03 and 0.30.
SEED = 42
EPOCHS = 5
BATCH_SIZE = 32
LEARNING_RATE = 0.001
IMAGE_SIZE = 224
VAL_FRACTION = 0.20

SAVE_TO_DRIVE = True
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    results_root = Path("/content/drive/MyDrive/plant-disease-classifier/results")
else:
    results_root = PROJECT_DIR / "results"
RUN_DIR = results_root / datetime.now(timezone.utc).strftime("demo_%Y%m%d_%H%M%S")
RUN_DIR.mkdir(parents=True)
print("Results:", RUN_DIR)

## 3. Inspect metadata and split the data

The pinned Hugging Face color file lists and leaf map provide image records without executing `plant_village.py`, the dataset's custom Python loading script. Sorting class names gives both models the same label mapping.

The metadata audit found 227 leaf identifiers shared across the published splits. Excluding the 290 matching training images preserves all 10,709 test records. The remaining training data is split by leaf group within each class, followed by selection of a fraction of the training groups. Fractions are approximate in image counts because groups have different numbers of photos. Validation stays fixed across model initializations and fractions for the same seed.

The upstream map uses filename fallbacks for some images, so physical-leaf separation cannot be guaranteed for unidentified leaves. Test metadata supports overlap checks only; test images remain uninspected and unevaluated.

In [ ]:
metadata = load_metadata(CACHE_DIR)
classes = metadata["classes"]
splits = make_splits(metadata, fraction=FRACTION, seed=SEED, val_fraction=VAL_FRACTION)
print("Published splits:", metadata["published_counts"])
print("Classes:", len(classes), classes)
print("Development counts:", splits["counts"])
print("Actual image fraction:", splits["actual_image_fraction"])
(RUN_DIR / "splits.json").write_text(json.dumps(splits, indent=2))
settings = {"fraction": FRACTION, "seed": SEED, "epochs": EPOCHS,
            "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
            "image_size": IMAGE_SIZE, "val_fraction": VAL_FRACTION,
            "device": DEVICE, "dataset_revision": metadata["revision"],
            "torch_version": str(torch.__version__), "torchvision_version": torchvision.__version__}
(RUN_DIR / "settings.json").write_text(json.dumps(settings, indent=2))

### Reproduce the metadata audit

These checks use the downloaded file lists and leaf map, without opening images. The first cell counts records, duplicate paths and leaf identifiers shared by the published splits. Shared identifiers explain the conservative training exclusions; they do not visually prove that two photos show the same leaf.

The second cell checks the selected training and validation splits. All three overlap counts should be zero, and both splits should contain every class. A failed assertion stops execution so the split can be investigated before training.

In [ ]:
train_records = metadata["records"]["train"]
test_records = metadata["records"]["test"]
print("Metadata files:", CACHE_DIR / "metadata")
print("Example training record:", train_records[0])
print("Example class:", classes[train_records[0]["label"]])
print("Training images:", len(train_records))
print("Test images:", len(test_records))
print("Classes:", len(classes))

train_paths = [row["image_path"] for row in train_records]
test_paths = [row["image_path"] for row in test_records]
print("Duplicate training paths:", len(train_paths) - len(set(train_paths)))
print("Duplicate test paths:", len(test_paths) - len(set(test_paths)))

train_leaf_ids = {row["leaf_id"] for row in train_records}
test_leaf_ids = {row["leaf_id"] for row in test_records}
shared_leaf_ids = train_leaf_ids & test_leaf_ids
print("Leaf IDs shared by published train/test:", len(shared_leaf_ids))
print("Training images with shared leaf IDs:",
      sum(row["leaf_id"] in shared_leaf_ids for row in train_records))

In [ ]:
selected_train_ids = {train_records[i]["leaf_id"] for i in splits["train_indices"]}
validation_ids = {train_records[i]["leaf_id"] for i in splits["validation_indices"]}
print("Selected train/validation overlap:", len(selected_train_ids & validation_ids))
print("Selected train/test overlap:", len(selected_train_ids & test_leaf_ids))
print("Validation/test overlap:", len(validation_ids & test_leaf_ids))

assert not (selected_train_ids & validation_ids), "Training and validation share leaf IDs"
assert not (selected_train_ids & test_leaf_ids), "Training and test share leaf IDs"
assert not (validation_ids & test_leaf_ids), "Validation and test share leaf IDs"

training_classes = {train_records[i]["label"] for i in splits["train_indices"]}
validation_classes = {train_records[i]["label"] for i in splits["validation_indices"]}
print("Classes in selected training subset:", len(training_classes))
print("Classes in validation:", len(validation_classes))
assert training_classes == validation_classes == set(range(len(classes))), "A split is missing classes"

## 4. Download development images

The source archive is about 2.18 GB and contains all image configurations. It is cached outside the repository, with only selected training and validation color images extracted. Additional disk space is needed for the extracted images and dependencies. Test images remain unopened.

In [ ]:
data = load_images(metadata, splits, CACHE_DIR)
print(data)

## 5. Visually inspect training images

The gallery shows one sampled training image per class before resizing or normalization, with its label and original dimensions. This allows a quick check of image quality and possible label issues. Changing `SAMPLE_SEED` displays another sample. Validation and test images are excluded.

In [ ]:
import random
import matplotlib.pyplot as plt
from IPython.display import display

SAMPLE_SEED = SEED
sample_rng = random.Random(SAMPLE_SEED)
training_indices = set(splits["train_indices"])
training_rows = [i for i, source in enumerate(data["source_index"]) if source in training_indices]
labels = data["label"]
sample_rows = []
for label in range(len(classes)):
    candidates = [i for i in training_rows if labels[i] == label]
    sample_rows.append(sample_rng.choice(candidates))

for start in range(0, len(sample_rows), 8):
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for axis in axes.flat:
        axis.axis("off")
    for axis, index in zip(axes.flat, sample_rows[start:start + 8]):
        row = data[index]
        image = row["image"]
        label = classes[row["label"]]
        axis.imshow(image)
        title = label.replace("___", "\n")
        axis.set_title(f"{title}\n{image.width}×{image.height}, {image.mode}", fontsize=8)
    fig.tight_layout()
    display(fig)
    plt.close(fig)

## 6. Train and evaluate both models

Resetting the seed and rebuilding the training loader gives both runs the same batch order. Both models use full fine-tuning, cross-entropy and identical Adam settings. Checkpoint selection uses the highest validation accuracy, breaking ties by validation loss.

The selected checkpoints are evaluated on the same validation set using accuracy and macro F1 across all classes. These are development scores because validation also supports checkpoint selection. Saved outputs include losses, metrics, checkpoints, prediction labels and figures.

In [ ]:
import csv

comparison = []
for name, pretrained in [("random", False), ("pretrained", True)]:
    seed_everything(SEED)
    train_loader, val_loader = make_loaders(data, splits, batch_size=BATCH_SIZE,
                                           seed=SEED, image_size=IMAGE_SIZE)
    model = build_model(len(classes), pretrained=pretrained)
    output_dir = RUN_DIR / name
    history = train_model(model, train_loader, val_loader, classes, output_dir,
                          epochs=EPOCHS, learning_rate=LEARNING_RATE,
                          device=DEVICE, image_size=IMAGE_SIZE)
    del model
    model, checkpoint = load_model(output_dir / "best.pt", DEVICE)
    metrics, actual, predicted = evaluate(model, val_loader, len(classes))
    metrics["best_epoch"] = checkpoint["epoch"]
    (output_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))
    comparison.append({"model": name, **metrics})
    plot_history(history, output_dir / "learning_curves.png")
    plot_predictions(val_loader.dataset, actual, predicted, classes, output_dir / "examples.png")
    has_mistakes = plot_predictions(val_loader.dataset, actual, predicted, classes,
                                   output_dir / "mistakes.png", mistakes_only=True)
    if not has_mistakes:
        (output_dir / "mistakes.txt").write_text("No validation misclassifications in this run.")
    with (output_dir / "predictions.csv").open("w", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(["image_path", "actual", "predicted"])
        writer.writerows(zip(val_loader.dataset.records["image_path"],
                             [classes[i] for i in actual], [classes[i] for i in predicted]))
    del model

with (RUN_DIR / "comparison.csv").open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(comparison[0]))
    writer.writeheader()
    writer.writerows(comparison)

## 7. Review results

Scores and learning curves can be compared after training completes. Correct predictions and mistakes provide examples for discussing class imbalance, the short training budget and the limitations of controlled-background images. PlantVillage validation scores do not establish reliability on field images.

In [ ]:
from IPython.display import display, Markdown, Image as DisplayImage

table = "| Model | Accuracy | Macro F1 | Best epoch |\n|---|---:|---:|---:|\n"
for row in comparison:
    table += f"| {row['model']} | {row['accuracy']:.4f} | {row['macro_f1']:.4f} | {row['best_epoch']} |\n"
(RUN_DIR / "comparison.md").write_text(table)
display(Markdown(table))
for name in ("random", "pretrained"):
    display(Markdown(f"### {name.title()} ResNet-18"))
    for filename in ("learning_curves.png", "examples.png", "mistakes.png"):
        path = RUN_DIR / name / filename
        if path.exists():
            display(DisplayImage(filename=str(path)))
    note = RUN_DIR / name / "mistakes.txt"
    if note.exists():
        print(note.read_text())

## 8. Try inference

The inference example uses a validation image, showing its actual class alongside the prediction. The checkpoint restores the label mapping and image size. Confidence is a softmax score, not a calibrated probability. `predict_image` also accepts a local image path.

In [ ]:
model, checkpoint = load_model(RUN_DIR / "pretrained/best.pt", DEVICE)
example = val_loader.dataset.records[0]
display(example["image"])
print("Actual:", classes[example["label"]])
print("Prediction:", predict_image(example["image"], model, checkpoint["classes"], checkpoint["image_size"]))

## Next experiments

I plan to repeat the comparison with 1%, 3%, 10% and 30% of training leaf groups and multiple seeds. Validation will stay fixed across fractions for each seed, with test evaluation reserved for the final protocol. Classifier placement experiments are deferred.

The complete run directory in Drive provides the progress-report evidence: settings, split indices, two best checkpoints, losses, validation metrics and figures. Completed histories and checkpoints remain available after an interruption, but automatic resume is not implemented. Repeated training starts in a fresh run directory.